# Salary Drivers in the Indian Tech Job Market

Models disclosed salary (₹) as a function of **city, role, experience, and skill breadth** across the disclosed-salary subset of the JobAtlas index, then clusters co-occurring skills into skill sets via **K-means**.

**Data:** `marts.mart_salary_model_input` — 669 postings that disclosed a structured INR salary range. Salary disclosure is uncommon in the Indian market, so this is a deliberately filtered slice of the 9,000+ indexed postings, not the whole index. Salary is the range midpoint (`salary_mid`), in lakhs per annum (LPA).

**Stack:** pandas, scikit-learn, statsmodels. An R/tidyverse counterpart lives at `analysis/r/salary_regression.Rmd` (same regression, same data).

> Requires `DATABASE_URL` in the environment for the K-means cell (it reads `staging.jobs`). Before launching Jupyter: `set -a; source .env; set +a`.

In [1]:
import ast
import os
from collections import Counter

import numpy as np
import pandas as pd

pd.set_option("display.float_format", lambda v: f"{v:,.2f}")

## 1. Load the disclosed-salary dataset

Same CSV the R notebook reads, regenerated by `analysis/export_data.py`.

In [2]:
df = pd.read_csv("../data/salary_model_input.csv")
df = df.dropna(subset=["salary_mid"]).reset_index(drop=True)
df["salary_lpa"] = df["salary_mid"].astype(float) / 1e5  # ₹ lakhs per annum
print(df.shape)
df.head()

(667, 14)


,id,salary_min,salary_max,salary_mid,city,state,country,source,role,experience_min,experience_band,skill_count,posted_date,salary_lpa
0,4241,"1,800,000.00","2,600,000.00","2,200,000.00",Richmond Town,Karnataka,IN,adzuna,Data Engineer,NaN,Not specified,4,2026-05-29,22.00
1,4253,"2,400,000.00","2,800,000.00","2,600,000.00",Bangalore,Karnataka,IN,adzuna,Other,NaN,Not specified,0,2026-05-30,26.00
2,4254,"1,800,000.00","2,400,000.00","2,100,000.00",Bangalore,Karnataka,IN,adzuna,Other,NaN,Not specified,0,2026-05-30,21.00
3,4266,"3,500,000.00","4,000,000.00","3,750,000.00",Richmond Town,Karnataka,IN,adzuna,Business Analyst,7.00,Senior (6-10),0,2026-05-30,37.50
4,4373,"800,000.00","1,100,000.00","950,000.00",Bangalore,Karnataka,IN,adzuna,Other,NaN,Not specified,0,2026-05-30,9.50


## 2. Feature engineering — canonical metro

Adzuna stores the most-specific locality in `city` (e.g. *Richmond Town*), so fold known metros together using the **same mapping as the live salary explorer**, so the two artifacts agree. Anything that isn't a known metro (including missing values) falls to `Other` (kept, not dropped, to preserve n).

In [3]:
def to_metro(city) -> str:
    if not isinstance(city, str):
        return "Other"
    c = city.lower()
    if "bangalore" in c or "bengaluru" in c:
        return "Bangalore"
    if "mumbai" in c:
        return "Mumbai"
    if "delhi" in c:
        return "Delhi"
    if "gurgaon" in c or "gurugram" in c:
        return "Gurugram"
    if "noida" in c:
        return "Noida"
    if "pune" in c:
        return "Pune"
    if "hyderabad" in c:
        return "Hyderabad"
    if "chennai" in c:
        return "Chennai"
    if "kolkata" in c:
        return "Kolkata"
    if "ahmedabad" in c:
        return "Ahmedabad"
    return "Other"


df["city_metro"] = df["city"].map(to_metro)
df["experience_band"] = df["experience_band"].fillna("Not specified")
df["role"] = df["role"].fillna("Other")
df["skill_count"] = pd.to_numeric(df["skill_count"], errors="coerce").fillna(0).astype(int)
df["city_metro"].value_counts()

city_metro
Other        165
Delhi        128
Mumbai        88
Bangalore     80
Noida         56
Kolkata       41
Chennai       40
Pune          29
Hyderabad     22
Gurugram      17
Ahmedabad      1
Name: count, dtype: int64

## 3. Regression — salary ~ metro + role + experience + skill breadth

scikit-learn pipeline (one-hot encode the categoricals, then linear regression), with a held-out test split and 5-fold CV given the small n.

In [4]:
from sklearn.compose import ColumnTransformer
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_absolute_error, r2_score
from sklearn.model_selection import cross_val_score, train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder

cat = ["city_metro", "role", "experience_band"]
num = ["skill_count"]
X = df[cat + num]
y = df["salary_lpa"]

pre = ColumnTransformer(
    [("cat", OneHotEncoder(handle_unknown="ignore"), cat)],
    remainder="passthrough",
)
model = Pipeline([("pre", pre), ("lr", LinearRegression())])

X_tr, X_te, y_tr, y_te = train_test_split(X, y, test_size=0.2, random_state=42)
model.fit(X_tr, y_tr)
pred = model.predict(X_te)

print(f"Test R^2     : {r2_score(y_te, pred):.3f}")
print(f"Test MAE     : {mean_absolute_error(y_te, pred):.2f} LPA")
cv = cross_val_score(model, X, y, cv=5, scoring="r2")
print(f"5-fold CV R^2: {cv.mean():.3f} +/- {cv.std():.3f}")

Test R^2     : 0.145
Test MAE     : 7.72 LPA
5-fold CV R^2: 0.053 +/- 0.103


### Coefficient table with significance (statsmodels OLS)

Same design matrix, fit with statsmodels for standard errors and p-values.

In [5]:
import statsmodels.api as sm

Xd = pd.get_dummies(df[cat], drop_first=True).astype(float)
Xd["skill_count"] = df["skill_count"].astype(float)
Xd = sm.add_constant(Xd)

ols = sm.OLS(y, Xd).fit()
print(ols.summary())

                            OLS Regression Results                            
Dep. Variable:             salary_lpa   R-squared:                       0.190
Model:                            OLS   Adj. R-squared:                  0.155
Method:                 Least Squares   F-statistic:                     5.355
Date:                Sat, 06 Jun 2026   Prob (F-statistic):           2.21e-16
Time:                        20:02:47   Log-Likelihood:                -2427.7
No. Observations:                 667   AIC:                             4913.
Df Residuals:                     638   BIC:                             5044.
Df Model:                          28                                         
Covariance Type:            nonrobust                                         
                                    coef    std err          t      P>|t|      [0.025      0.975]
-------------------------------------------------------------------------------------------------
const         

## 4. K-means — clustering co-occurring skills into skill sets

Build a skill x skill co-occurrence matrix from posting skill arrays (`staging.jobs.skills_array`), then cluster the most frequent skills. Each cluster is a co-occurring "skill set" (e.g. a data-engineering stack vs. a frontend stack).

In [6]:
from sklearn.cluster import KMeans
from sklearn.preprocessing import normalize
from sqlalchemy import create_engine, text

engine = create_engine(os.environ["DATABASE_URL"])
with engine.connect() as conn:
    raw = pd.read_sql(
        text(
            "select skills as skills_array from staging.jobs "
            "where skills is not null and is_active and not is_duplicate"
        ),
        conn,
    )


def as_list(v):
    if isinstance(v, list):
        return [str(s).strip().lower() for s in v if s]
    if isinstance(v, str):
        try:
            return [str(s).strip().lower() for s in ast.literal_eval(v)]
        except Exception:
            return [t.strip().lower() for t in v.strip("{}").split(",") if t.strip()]
    return []


postings = raw["skills_array"].map(as_list)
freq = Counter(s for lst in postings for s in lst)
top = [s for s, _ in freq.most_common(40)]
idx = {s: i for i, s in enumerate(top)}

M = np.zeros((len(top), len(top)))
for lst in postings:
    present = [idx[s] for s in set(lst) if s in idx]
    for a in present:
        for b in present:
            if a != b:
                M[a, b] += 1

k = 6
km = KMeans(n_clusters=k, random_state=42, n_init=10)
labels = km.fit_predict(normalize(M))

for c in range(k):
    members = [top[i] for i in range(len(top)) if labels[i] == c]
    print(f"Cluster {c} ({len(members)} skills): {', '.join(members)}")

Cluster 0 (3 skills): c++, ruby, golang
Cluster 1 (8 skills): etl, spark, r, snowflake, power bi, tableau, scala, hadoop
Cluster 2 (6 skills): sql, nlp, django, tensorflow, pytorch, fastapi
Cluster 3 (16 skills): python, java, aws, azure, ci/cd, go, gcp, kubernetes, docker, git, terraform, kafka, mongodb, postgresql, mysql, bash
Cluster 4 (2 skills): elt, bigquery
Cluster 5 (5 skills): react, spring, javascript, node.js, typescript


## Notes

- This is a **descriptive** model over the current index snapshot of a **pre-launch** product — coefficients describe the disclosed-salary postings, not the whole market.
- City is folded to the same canonical metros as the live `/salary` explorer so the figures reconcile across artifacts.